# 02_clean: 数据清洗

**学生**：谢婧怡 25210094

说明：对 `dshw-p01/data/stock` 中的原始 CSV 执行缺失值检测、类型转换、日期处理、重复值与离群值标注，完成宽表/长表转换并保存为 CSV 与 Parquet。每段代码单元前均有简要说明。

## 单表清洗：读取并清洗每只股票的日度行情（缺失值检测、类型转换、重复值、离群值标注）
下面的单元会遍历 `dshw-p01/data/stock` 下的 CSV 文件，执行清洗，并将结果合并为长表 `df_all`。

In [10]:
import os
from datetime import datetime
import pandas as pd
import numpy as np

# 智能定位项目目录：优先使用相对的 'dshw-p01'，若不存在则在 workspace 下搜索含 'data/stock' 的目录（支持嵌套）
base_name = '.'
project_dir = None
# 首先尝试直接路径
if os.path.exists(base_name):
    project_dir = base_name
# 然后尝试常见工作区位置
if project_dir is None:
    workspace_root = os.path.join(os.path.expanduser('~'), 'Documents', 'GitHub', 'homework')
    if os.path.exists(workspace_root):
        for root, dirs, files in os.walk(workspace_root):
            if os.path.isdir(os.path.join(root, 'data', 'stock')):
                project_dir = root
                break
# 最后尝试在当前目录向上查找
if project_dir is None:
    p = os.getcwd()
    for _ in range(6):
        candidate = os.path.join(p, base_name)
        if os.path.exists(candidate):
            project_dir = candidate
            break
        p = os.path.dirname(p)

if project_dir is None:
    print('Could not find dshw-p01 folder or data/stock under the workspace.')
    print('Detected current working directory:', os.getcwd())
    print('Please run `01_download.ipynb` first, or set `project_dir` to the correct path, e.g.:')
    print(r"project_dir = r'C:\Users\Lenovo\Documents\GitHub\homework\dshw-p01\dshw-p01'")
    stock_dir = os.path.join(base_name, 'data', 'stock')
    files = []
else:
    stock_dir = os.path.join(project_dir, 'data', 'stock')
    if os.path.exists(stock_dir):
        files = [f for f in os.listdir(stock_dir) if f.endswith('.csv')]
    else:
        # 在 project_dir 中递归查找符合 stock_*.csv 的文件
        files = []
        for root, dirs, filenames in os.walk(project_dir):
            for fn in filenames:
                if fn.startswith('stock_') and fn.endswith('.csv'):
                    rel = os.path.relpath(os.path.join(root, fn), start=project_dir)
                    files.append(rel)

clean_dir = os.path.join(project_dir if project_dir else base_name, 'data', 'clean')
os.makedirs(clean_dir, exist_ok=True)

print('Using project_dir:', project_dir)
print('Found', len(files), 'stock files')
records = []
summary_rows = []
for f in files:
    # 支持直接在 stock_dir 下的文件名或递归发现的相对路径
    if os.path.isabs(f):
        path = f
    else:
        if f.startswith('stock_'):
            path = os.path.join(project_dir if project_dir else base_name, 'data', 'stock', f)
        else:
            path = os.path.join(project_dir if project_dir else base_name, f)
    code = os.path.basename(f).replace('stock_', '').replace('.csv','')
    try:
        df = pd.read_csv(path, dtype=str)
    except Exception as e:
        print('Failed to read', f, e)
        continue
    # 标准化列名为小写并去空格
    df.columns = [c.strip().lower() for c in df.columns]
    # 尝试识别日期列常见名
    date_col = None
    for cand in ['日期','date','trade_date','time']:
        if cand in df.columns:
            date_col = cand
            break
    if date_col is None:
        print('No date column for', f)
        continue
    # 日期解析
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    # 排序并设索引
    df = df.sort_values(date_col).reset_index(drop=True)
    df = df.rename(columns={date_col: 'date'})
    # 确保数值列
    num_cols = []
    for col in ['open','close','high','low','volume','amount']:
        if col in df.columns:
            num_cols.append(col)
            df[col] = pd.to_numeric(df[col], errors='coerce')
    # 缺失值统计
    miss_count = df.isnull().sum()
    miss_pct = df.isnull().mean()
    summary_rows.append({'code':code, 'rows':len(df), 'missing_counts': miss_count.to_dict(), 'missing_pct': miss_pct.to_dict()})
    # 缺失值处理：采用向前填充（ffill），金融时间序列停牌时向前填充通常合理；若仍有缺失则删除
    df = df.ffill()
    df = df.dropna()
    # 删除重复行
    before = len(df)
    df = df.drop_duplicates()
    # 计算日收益率并标注极端值（±20%）
    if 'close' in df.columns:
        df = df.sort_values('date')
        df['pct_chg'] = df['close'].pct_change()
        df['is_extreme'] = df['pct_chg'].abs() > 0.20
    else:
        df['pct_chg'] = np.nan
        df['is_extreme'] = False
    # 添加 code 列
    df['code'] = code
    # 记录并保存单只股票清洗结果（长表）
    out_path = os.path.join(clean_dir, f'stock_{code}_clean.csv')
    df.to_csv(out_path, index=False, encoding='utf-8-sig')
    records.append(df[['date','code'] + [c for c in ['open','close','high','low','volume','amount','pct_chg','is_extreme'] if c in df.columns]])
    print('Cleaned', code, 'rows', len(df))

# 合并为长表 df_all
if records:
    df_all = pd.concat(records, ignore_index=True)
else:
    df_all = pd.DataFrame()

print('Combined rows:', len(df_all))


Using project_dir: .
Found 10 stock files
Cleaned 000002 rows 1514
Cleaned 000625 rows 1514
Cleaned 000858 rows 1514
Cleaned 002352 rows 1511
Cleaned 600028 rows 1514
Cleaned 600050 rows 1514
Cleaned 600104 rows 1514
Cleaned 600519 rows 1514
Cleaned 601398 rows 1514
Cleaned 601988 rows 1514
Combined rows: 15137


## 宽表与长表转换
将 `df_all` 的收盘价合并为宽表（每列一只股票），再用 `pd.melt` 转回长表以演示两种格式的使用场景。

In [11]:
# 宽表：pivot date x code -> close
if not df_all.empty and 'close' in df_all.columns:
    wide = df_all.pivot_table(index='date', columns='code', values='close')
    wide_path = os.path.join(clean_dir, 'stock_close_wide.csv')
    wide.to_csv(wide_path, encoding='utf-8-sig')
    print('Saved wide table to', wide_path)
    # melt 回长表
    long_from_wide = wide.reset_index().melt(id_vars='date', var_name='code', value_name='close')
    print('Wide->Long rows:', len(long_from_wide))
else:
    print('df_all empty or no close column; skipping wide/long conversion')

Saved wide table to .\data\clean\stock_close_wide.csv
Wide->Long rows: 15140


## 多表合并：与指数和宏观数据合并
将长表与指数按日期合并；将月度宏观数据映射到每个交易日所属月份再合并。

In [12]:
# 保证有可用的 `base` 变量：优先使用前面单元的 `project_dir`，否则回退到默认名
base = globals().get('project_dir') if globals().get('project_dir') else 'dshw-p01'

# 读取指数数据（手动下载的沪深300，列名为 Idxtrd01, Idxtrd05 等）
index_path = os.path.join(base, 'data', 'index', 'index_000300.csv')
if os.path.exists(index_path):
    # 指定编码为 gbk 解决中文环境导出文件的编码问题
    idx = pd.read_csv(index_path, encoding='gbk')
    # 列名统一小写
    idx.columns = [c.strip().lower() for c in idx.columns]
    # 根据手动下载文件的实际列名映射
    # 常见列名: indexcd, idxtrd01, idxtrd02, idxtrd03, idxtrd04, idxtrd05, idxtrd06, idxtrd07, idxtrd08, idxtrd09
    # idxtrd01 = 日期, idxtrd05 = 收盘
    if 'idxtrd01' in idx.columns and 'idxtrd05' in idx.columns:
        idx['date'] = pd.to_datetime(idx['idxtrd01'], format='%Y/%m/%d', errors='coerce')
        idx['index_close'] = pd.to_numeric(idx['idxtrd05'], errors='coerce')
        # 可选：同时保存开盘、最高、最低等（后续分析可能用到）
        if 'idxtrd02' in idx.columns:
            idx['open'] = pd.to_numeric(idx['idxtrd02'], errors='coerce')
        if 'idxtrd03' in idx.columns:
            idx['high'] = pd.to_numeric(idx['idxtrd03'], errors='coerce')
        if 'idxtrd04' in idx.columns:
            idx['low'] = pd.to_numeric(idx['idxtrd04'], errors='coerce')
        if 'idxtrd06' in idx.columns:
            idx['volume'] = pd.to_numeric(idx['idxtrd06'], errors='coerce')
        if 'idxtrd07' in idx.columns:
            idx['amount'] = pd.to_numeric(idx['idxtrd07'], errors='coerce')
        # 删除空日期或空收盘价的记录
        idx = idx.dropna(subset=['date', 'index_close'])
        idx = idx.sort_values('date')
        print(f"Index loaded: {len(idx)} rows, date range {idx['date'].min()} to {idx['date'].max()}")
    else:
        # 如果列名不匹配，尝试自动推断（备用）
        print("Manual index columns not found, trying auto inference...")
        date_found = False
        for col in idx.columns:
            try:
                parsed = pd.to_datetime(idx[col], errors='coerce')
                if parsed.notna().sum() > 0:
                    idx['date'] = parsed
                    date_found = True
                    print(f"Inferred date column from '{col}'")
                    break
            except Exception:
                continue
        if not date_found:
            print("Could not find date column in index file; skipping index processing")
            idx = pd.DataFrame()
        else:
            # 找数值列作为收盘价
            close_candidates = ['index_close','close','close_price','last','price','收盘价','idxtrd05']
            found_close = False
            for cand in close_candidates:
                if cand in idx.columns:
                    idx['index_close'] = pd.to_numeric(idx[cand], errors='coerce')
                    found_close = True
                    break
            if not found_close:
                # 尝试找第一个全数值的列
                for col in idx.columns:
                    if col != 'date' and pd.to_numeric(idx[col], errors='coerce').notna().any():
                        idx['index_close'] = pd.to_numeric(idx[col], errors='coerce')
                        found_close = True
                        print(f"Using column '{col}' as index_close")
                        break
            if not found_close:
                idx['index_close'] = pd.NA
                print("Warning: no close column found; index_close set to NA")
            idx = idx.dropna(subset=['date'])
            idx = idx.sort_values('date')
else:
    idx = pd.DataFrame()
    print('Index file not found:', index_path)

# 读取宏观 CPI（若存在），并映射到每个交易日的月份
macro_path = os.path.join(base, 'data', 'macro', 'macro_cpi.csv')
if os.path.exists(macro_path):
    macro = pd.read_csv(macro_path)
    macro.columns = [c.strip().lower() for c in macro.columns]
    # 试图找到日期与值列
    date_col = None
    val_col = None
    for cand in ['日期','date','time','month']:
        if cand in macro.columns:
            date_col = cand
            break
    for cand in ['value','cpi','同比','同比增速','year_on_year']:
        if cand in macro.columns:
            val_col = cand
            break
    if date_col is not None and val_col is not None:
        macro['date'] = pd.to_datetime(macro[date_col], errors='coerce')
        macro = macro.sort_values('date')
        macro = macro.rename(columns={val_col: 'cpi_value'})
        macro['month'] = macro['date'].dt.to_period('M').astype(str)
        print('Macro rows:', len(macro))
    else:
        # 尝试自动推断日期和值列
        inferred_date = None
        inferred_val = None
        for col in macro.columns:
            try:
                parsed = pd.to_datetime(macro[col], errors='coerce')
                if parsed.notna().sum() > 0:
                    inferred_date = col
                    macro['date'] = parsed
                    break
            except Exception:
                continue
        if inferred_date is not None:
            # 选择非日期的第一个数值列作为值
            for col in macro.columns:
                if col == inferred_date:
                    continue
                try:
                    # try convert to numeric removing %
                    ser = macro[col].astype(str).str.replace('%','',regex=False)
                    ser = ser.str.replace(',','',regex=False)
                    if ser.str.replace('.','',regex=False).str.isnumeric().any():
                        inferred_val = col
                        break
                except Exception:
                    continue
        if inferred_date is not None and inferred_val is not None:
            macro = macro.sort_values('date')
            macro = macro.rename(columns={inferred_val: 'cpi_value'})
            macro['month'] = macro['date'].dt.to_period('M').astype(str)
            print(f"Inferred macro date='{inferred_date}' val='{inferred_val}' rows={len(macro)}")
        else:
            macro = pd.DataFrame()
            print('Macro file found but could not infer columns')
else:
    macro = pd.DataFrame()
    print('Macro file not found:', macro_path)

# 将长表与指数合并（left join）
if 'df_all' in globals() and not df_all.empty and not idx.empty:
    before = len(df_all)
    # 确保 idx 有 date 和 index_close 列
    if 'date' in idx.columns and 'index_close' in idx.columns:
        merged = pd.merge(df_all, idx[['date','index_close']], on='date', how='left')
        after = len(merged)
        print('Merged with index: before', before, 'after', after)
    else:
        print('Index missing required columns for merge; skipping index merge')
        merged = df_all.copy()
else:
    merged = df_all.copy() if 'df_all' in globals() and not df_all.empty else pd.DataFrame()
    print('Skipped index merge')

# 将宏观按月份映射到交易日并合并
if not merged.empty and not macro.empty:
    merged['month'] = merged['date'].astype('datetime64[ns]').dt.to_period('M').astype(str)
    before = len(merged)
    merged = pd.merge(merged, macro[['month','cpi_value']], on='month', how='left')
    after = len(merged)
    print('Merged with macro: before', before, 'after', after)
else:
    print('Skipped macro merge')


Index loaded: 1513 rows, date range 2020-01-02 00:00:00 to 2026-04-02 00:00:00
Inferred macro date='日期' val='今值' rows=477
Merged with index: before 15137 after 15137
Merged with macro: before 15137 after 15317


## 保存清洗后数据（CSV 与 Parquet）
将清洗后的长表保存为 `data/clean/stock_clean.csv`，并以 Parquet 格式保存一份 `data/clean/stock_clean.parquet`。

In [13]:
combined_dir = os.path.join(base, 'data', 'combined')
os.makedirs(combined_dir, exist_ok=True)
# 保存长表 CSV
clean_out = os.path.join(clean_dir, 'stock_clean.csv')
if not merged.empty:
    merged.to_csv(clean_out, index=False, encoding='utf-8-sig')
    print('Saved cleaned long table to', clean_out)
    # 保存 Parquet 作为进阶格式
    parquet_out = os.path.join(clean_dir, 'stock_clean.parquet')
    try:
        merged.to_parquet(parquet_out, index=False)
        print('Saved parquet to', parquet_out)
    except Exception as e:
        print('Parquet save failed:', e)
    # 另存一份合并数据到 combined 文件夹
    combined_out = os.path.join(combined_dir, 'combined_data.csv')
    merged.to_csv(combined_out, index=False, encoding='utf-8-sig')
    print('Saved combined data to', combined_out)
else:
    print('No merged data to save')

Saved cleaned long table to .\data\clean\stock_clean.csv
Saved parquet to .\data\clean\stock_clean.parquet
Saved combined data to .\data\combined\combined_data.csv


## 小结（请在 Notebook 中补充说明）
- 缺失值处理采用 `ffill`，针对停牌情形合理；剩余缺失行被删除。
- 离群值仅作标注（`is_extreme`），不删除，便于后续人工判断。
- Parquet 文件用于演示列式存储优点；在小数据上与 CSV 差异不大。